# Exporting Simulation Results

`Simulation.run()` returns a `BatchResult`. This notebook shows the three
export helpers it carries and the common analytical patterns built on top
of them: a tidy long-form `DataFrame` (`.to_dataframe()`), a pivoted
wide-form table for plotting (`.to_wide()`), and file export
(`.to_csv()`/`.to_parquet()`) with round-trip precision guarantees.

Needs `pandas` and `pyarrow`: `pip install -e ".[export]"`

## 1  Build a small single-CV simulation

Sparged batch fermenter, aerobic growth on acetic acid -- same topology as
[`../templates/batch_fermenter.py`](../templates/batch_fermenter.py) but
constructed here from the factory helpers for brevity, since the topology
itself isn't the point of this notebook.

In [1]:
import tempfile
import pathlib

import numpy as np
import pandas as pd

from PyOMES.templates.stirred_tank import (
    StirredTankFactory,
    VesselConfig,
    TransferConfig,
    GasFeedConfig,
)
from PyOMES.core import Simulation, SimultaneousEulerSolver

cv = StirredTankFactory.create_volume(
    vessel=VesselConfig(V_total_L=10.0, T_K=305.15),
    transfer=TransferConfig.default_kinetic(kLa_O2=120.0),
    gas_feed=GasFeedConfig(vvm_min=0.5, composition={"O2": 0.21, "N2": 0.79}),
)

sim = Simulation(cvs={"main": cv}, solver=SimultaneousEulerSolver())

result = sim.run(tau_h=2.0, n_steps=200)

print(f"Run complete: {len(result.t_h)} time points, "
      f"runtime {result.runtime_s*1000:.1f} ms")
print("Gas species:   ", sorted(result.gas_mol["main"]))
print("Liquid species:", sorted(result.liquid_mol["main"]))

Run complete: 201 time points, runtime 26.7 ms
Gas species:    ['CO2', 'N2', 'O2']
Liquid species: ['CO2', 'N2', 'O2']


## 2  Long-form DataFrame

`.to_dataframe()` returns one row per (cv, phase/channel, species, time
point) -- the tidy shape most pandas operations expect.

In [2]:
df = result.to_dataframe()

print("Long-form DataFrame -- first 5 rows:")
print(df.head())
print(f"\nShape: {df.shape}  (rows x columns)")
print("channel_kind values:", sorted(df["channel_kind"].unique()))

Long-form DataFrame -- first 5 rows:
    t_h cv_key channel_kind phase_key species     value
0  0.00   main    phase_mol       gas      O2  0.016733
1  0.01   main    phase_mol       gas      O2  0.036861
2  0.02   main    phase_mol       gas      O2  0.055426
3  0.03   main    phase_mol       gas      O2  0.073581
4  0.04   main    phase_mol       gas      O2  0.091630

Shape: (1809, 6)  (rows x columns)
channel_kind values: ['P_atm', 'ionic_strength', 'pH', 'phase_mol']


## 3  Groupby aggregation across species

Mean value of each channel_kind across the whole run -- the standard
pandas aggregation pattern applies directly to the long-form table.

In [3]:
summary = (
    df.groupby(["cv_key", "channel_kind", "species"], dropna=False)["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "mean_value"})
)
print("Per-channel mean values (all CVs):")
print(summary.to_string(index=False))

Per-channel mean values (all CVs):
cv_key   channel_kind species  mean_value
  main          P_atm     NaN  113.175240
  main ionic_strength     NaN         NaN
  main             pH     NaN         NaN
  main      phase_mol     CO2    0.000062
  main      phase_mol      N2    3.819379
  main      phase_mol      O2    1.015747


## 4  Wide-form for plotting

`.to_wide()` pivots the liquid `phase_mol` rows into a `t_h x species`
DataFrame -- the shape plotting libraries expect. The manual pivot below
reconstructs it from the long form, to show the pattern for multi-CV or
cross-phase queries `.to_wide()` doesn't cover directly.

In [4]:
liquid_wide = result.to_wide("main", "liquid")
print("Wide-form liquid mol (first 3 rows):")
print(liquid_wide.head(3))
print()

gas_wide = result.to_wide("main", "gas")
print("Wide-form gas mol (first 3 rows):")
print(gas_wide.head(3))
print()

# Demonstrate: reconstruct a wide table manually from the long form
manual_wide = (
    df.query("cv_key == 'main' and channel_kind == 'phase_mol' and phase_key == 'liquid'")
    .pivot(index="t_h", columns="species", values="value")
    .rename_axis(None, axis="columns")
)
pd.testing.assert_frame_equal(liquid_wide, manual_wide)
print("Manual pivot matches to_wide() -- OK")

Wide-form liquid mol (first 3 rows):
           CO2        N2        O2
t_h                               
0.00  0.000092  0.003709  0.001967
0.01  0.000092  0.003709  0.001967
0.02  0.000092  0.007912  0.003530

Wide-form gas mol (first 3 rows):
           CO2        N2        O2
t_h                               
0.00  0.000032  0.063107  0.016733
0.01  0.000032  0.138827  0.036861
0.02  0.000032  0.210343  0.055426

Manual pivot matches to_wide() -- OK


## 5  CSV round-trip

CSV is lossy in principle (text formatting), but `.to_csv()` writes with
enough precision (`%.10e`, ~10 significant digits) that the round-trip
error stays far below any physically meaningful tolerance.

In [5]:
with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = pathlib.Path(tmpdir) / "run.csv"
    result.to_csv(str(csv_path))

    df_back = pd.read_csv(str(csv_path))
    print(f"CSV written: {csv_path.name}  ({csv_path.stat().st_size // 1024} KB)")
    print(f"CSV re-read: {df_back.shape[0]} rows")

    # Verify numerical round-trip within %.10e precision
    orig_vals = df["value"].dropna().to_numpy()
    back_vals = df_back["value"].dropna().to_numpy()
    # %.10e preserves ~10 significant decimal digits; relative error <~ 5e-11.
    # Use absolute tolerance of 1e-8 to cover values up to ~200 (P_atm).
    max_err = np.max(np.abs(orig_vals - back_vals))
    print(f"CSV max |value| round-trip error: {max_err:.2e}")
    assert max_err < 1e-8, f"CSV precision loss: {max_err}"

CSV written: run.csv  (92 KB)
CSV re-read: 1809 rows
CSV max |value| round-trip error: 4.96e-09


## 6  Parquet round-trip

Parquet is a binary columnar format: lossless (exact float64 round-trip),
and its columnar layout lets pandas read back only the columns you need --
useful for wide tables or very long runs.

In [6]:
with tempfile.TemporaryDirectory() as tmpdir:
    pq_path = pathlib.Path(tmpdir) / "run.parquet"
    result.to_parquet(str(pq_path))

    df_pq = pd.read_parquet(str(pq_path))
    print(f"Parquet written: {pq_path.name}  ({pq_path.stat().st_size // 1024} KB)")
    print(f"Parquet re-read: {df_pq.shape[0]} rows")

    # Parquet is lossless -- exact float64 equality
    orig_vals = df["value"].dropna().to_numpy()
    pq_vals = df_pq["value"].dropna().to_numpy()
    np.testing.assert_array_equal(orig_vals, pq_vals)
    print("Parquet round-trip: exact float64 equality -- OK")
    print()

    df_slim = pd.read_parquet(str(pq_path), columns=["t_h", "species", "value"])
    print(f"Selective Parquet read (3 columns): shape {df_slim.shape}")

print()
print("All export checks passed.")

Parquet written: run.parquet  (14 KB)
Parquet re-read: 1809 rows
Parquet round-trip: exact float64 equality -- OK

Selective Parquet read (3 columns): shape (1809, 3)

All export checks passed.
